In [ ]:
import json
import random
from PIL import Image
from torch.utils.data import Dataset
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import DataLoader
from transformers import CLIPProcessor
import torch.optim as optim
from tqdm import tqdm
from transformers import CLIPModel

# Dataset

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, jsonl_path, processor, split="train", train_ratio=0.8, seed=42):
        with open(jsonl_path, "r", encoding="utf-8") as f:
            data = [json.loads(line) for line in f]
        random.seed(seed)
        random.shuffle(data)

        n_train = int(len(data) * train_ratio)

        if split == "train":
            self.data = data[:n_train]
        elif split == "test":
            self.data = data[n_train:]
        elif split == "all":
            self.data = data

        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        caption = item["caption"]
        label = int(item["label"])
        inputs = self.processor(
            text=caption,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs, label

# Classifier

In [8]:
class CLIPClassifier(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", num_labels=2):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.clip.config.projection_dim, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(input_ids=input_ids,
                            attention_mask=attention_mask,
                            pixel_values=pixel_values)
        combined_embeds = (outputs.text_embeds + outputs.image_embeds) / 2
        logits = self.classifier(combined_embeds)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {"loss": loss, "logits": logits}

# Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

jsonl_path = "combined_dataset.jsonl"
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

train_dataset = ImageTextDataset(jsonl_path, processor, split="train")
test_dataset = ImageTextDataset(jsonl_path, processor, split="test")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = CLIPClassifier().to(device)
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

best_val_loss = float("inf")
patience = 1
epochs_no_improve = 0
epochs = 4

In [ ]:
for epoch in range(epochs):
    # Train
    model.train()
    total_loss = 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (Train)"):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} (Test)"):
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            outputs = model(**inputs, labels=labels)
            test_loss += outputs["loss"].item()
            preds = outputs["logits"].argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_test_loss = test_loss / len(test_loader)
    val_acc = correct / total

    print(f"Epoch {epoch+1} - Test Loss: {avg_test_loss:.3f}, Test Acc: {val_acc:.3f}")

    # Saving and Early Stopping
    if avg_test_loss < best_val_loss:
        best_val_loss = avg_test_loss
        torch.save(model.clip.state_dict(), "clip_best_model.pth")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered")
            break

Epoch 1/5 (Val): 100%|██████████| 38/38 [00:44<00:00,  1.16s/it]


Epoch 1 - Val Loss: 0.513, Val Acc: 0.832


Epoch 2/5 (Val): 100%|██████████| 38/38 [00:43<00:00,  1.13s/it]


Epoch 2 - Val Loss: 0.318, Val Acc: 0.970


Epoch 3/5 (Val): 100%|██████████| 38/38 [00:44<00:00,  1.16s/it]


Epoch 3 - Val Loss: 0.251, Val Acc: 0.977


Epoch 4/5 (Val): 100%|██████████| 38/38 [00:43<00:00,  1.15s/it]


Epoch 4 - Val Loss: 0.203, Val Acc: 0.983


Epoch 5/5 (Val): 100%|██████████| 38/38 [00:43<00:00,  1.15s/it]

Epoch 5 - Val Loss: 0.323, Val Acc: 0.886
